In [ ]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import pandas as pd
import pyarrow.parquet as pq
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

ModuleNotFoundError: No module named 'utils'

In [4]:
table = pq.read_table("../data/processed/product_documents.parquet")
docs = table.to_pandas()

In [5]:
docs.head()

,parent_asin,product_title,document
0,B0B9J2JWSX,ASUS TUF Gaming 23.8” 1080P Monitor (VG249Q1R)...,Computers ASUS TUF Gaming 23.8” 1080P Monitor ...
1,B0C2PVFRWV,"USB Type C Cable, Anker [2-Pack 6Ft] Premium N...","Cell Phones & Accessories USB Type C Cable, An..."
2,B01KIOU4EO,Echo Show - 1st Generation Black,Amazon Devices Echo Show - 1st Generation Blac...
3,B07GBYKWGW,ASUS AC1200 Dual Band WiFi Repeater & Range Ex...,Computers ASUS AC1200 Dual Band WiFi Repeater ...
4,B00O5VX2KK,"ASUS C300 ChromeBook 13.3 Inch (Intel Celeron,...",Computers ASUS C300 ChromeBook 13.3 Inch (Inte...


In [9]:
langchain_docs = [
    Document(
        page_content=row["document"],
        metadata={
            "parent_asin": row["parent_asin"],
            "product_title": row["product_title"]
        }
    )
    for _, row in docs.iterrows()
]

In [10]:
vectorstore = FAISS.from_documents(langchain_docs, embeddings)

In [12]:
results = vectorstore.similarity_search("bluetooth headphones", k=3)

for r in results:
    print(r.page_content)
    print(r.metadata)

All Electronics DOBOPO Wireless Earbuds Bluetooth 5.3 Headphones 50Hrs Playtime Sports Earphones Over-Ear Earhooks Headset with LED Display, ENC Mic, IP7 Waterproof for Workout, Running, Gym (2023 New) Bluetooth 5.3 and Effortless Pairing: Wireless headphones with advanced bluetooth 5.3 technology support HSP HFP A2DP AVRCP, owns fast and stable transmission without tangling, which makes the bluetooth earbuds connect to electronic devices more consistently, providing you with an uninterrupted calling and music experience and Noise cancellation.These wireless earphones would instant paired once taken out from the charging case and auto connects with the last-paired device. Superior Clear Call and High-fidelity Audio: These bluetooth headphones with 13mm speakers and triple-layer composite diaphragms provide powerful bass( lowest 16Hz), stunning treble(up to 20kHz) and clear mids. These earbud are designed for sound and music lovers. It can support the mono mode and twin stereo mode, you

In [14]:
vectorstore.save_local("langchain_semantic_index")